In [ ]:
# ============================================================
# CROP RECOMMENDATION MODEL TRAINING
# RANDOM FOREST CLASSIFIER
# ============================================================

import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report


# ============================================================
# PROJECT PATH
# ============================================================

BASE_DIR = Path.cwd()

# If notebook is running from the notebooks folder
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "crop_data.csv"
MODEL_DIR = BASE_DIR / "models"

MODEL_PATH = MODEL_DIR / "crop_model.pkl"
ENCODER_PATH = MODEL_DIR / "label_encoder.pkl"

MODEL_DIR.mkdir(parents=True, exist_ok=True)


print("=" * 60)
print("CROP RECOMMENDATION MODEL TRAINING")
print("=" * 60)

print("\nProject directory:")
print(BASE_DIR)

print("\nDataset:")
print(DATA_PATH)


# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# CHECK DATASET
# ============================================================

required_columns = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall",
    "label"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing columns in crop_data.csv: {missing_columns}"
    )


# Remove duplicate rows

df = df.drop_duplicates()

print("\nDataset after removing duplicates:")
print(df.shape)


# ============================================================
# HANDLE MISSING VALUES
# ============================================================

features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

target = "label"


for column in features:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    df[column] = df[column].fillna(
        df[column].median()
    )


df[target] = df[target].astype(str)


# ============================================================
# PREPARE FEATURES AND TARGET
# ============================================================

X = df[features]

y = df[target]

print("\nFeatures:")
print(features)

print("\nCrop classes:")
print(sorted(y.unique()))

print("\nNumber of crop classes:")
print(y.nunique())


# ============================================================
# ENCODE CROP LABELS
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("\nEncoded crop classes:")
print(label_encoder.classes_)


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

number_of_classes = len(label_encoder.classes_)
number_of_samples = len(df)

# For very small datasets, use a larger test set
# so that the test set can contain enough samples.

minimum_test_samples = number_of_classes

test_size = max(
    0.20,
    minimum_test_samples / number_of_samples
)

# Prevent test size from becoming too large
test_size = min(test_size, 0.40)

print("\nTest size:", test_size)


# Check whether stratified splitting is possible

class_counts = y_encoded

_, counts = np.unique(
    class_counts,
    return_counts=True
)

can_stratify = (
    counts.min() >= 2
)


if can_stratify:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=test_size,
        random_state=42,
        stratify=y_encoded
    )

else:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=test_size,
        random_state=42
    )


print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)


print("\nTraining Random Forest model...")

model.fit(
    X_train,
    y_train
)

print("Model training completed!")


# ============================================================
# MODEL PREDICTION
# ============================================================

y_pred = model.predict(X_test)


# ============================================================
# ACCURACY
# ============================================================

train_predictions = model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

test_accuracy = accuracy_score(
    y_test,
    y_pred
)


print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(
    f"\nTraining Accuracy: "
    f"{train_accuracy * 100:.2f}%"
)

print(
    f"Testing Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

try:

    print(
        classification_report(
            y_test,
            y_pred,
            labels=np.unique(y_test),
            target_names=[
                label_encoder.classes_[i]
                for i in np.unique(y_test)
            ],
            zero_division=0
        )
    )

except Exception as error:

    print(
        "Classification report could not be generated:"
    )

    print(error)


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame(
    {
        "Feature": features,
        "Importance": model.feature_importances_
    }
)

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
display(feature_importance)


# ============================================================
# SAVE RANDOM FOREST MODEL
# ============================================================

with open(
    MODEL_PATH,
    "wb"
) as file:

    pickle.dump(
        model,
        file
    )


# ============================================================
# SAVE LABEL ENCODER
# ============================================================

with open(
    ENCODER_PATH,
    "wb"
) as file:

    pickle.dump(
        label_encoder,
        file
    )


# ============================================================
# VERIFY SAVED FILES
# ============================================================

print("\n" + "=" * 60)
print("VERIFYING MODEL FILES")
print("=" * 60)

print(
    "\nModel file:",
    MODEL_PATH
)

print(
    "Model file size:",
    MODEL_PATH.stat().st_size,
    "bytes"
)

print(
    "\nEncoder file:",
    ENCODER_PATH
)

print(
    "Encoder file size:",
    ENCODER_PATH.stat().st_size,
    "bytes"
)


if MODEL_PATH.stat().st_size == 0:

    raise ValueError(
        "crop_model.pkl was created but is empty."
    )


if ENCODER_PATH.stat().st_size == 0:

    raise ValueError(
        "label_encoder.pkl was created but is empty."
    )


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("=" * 60)

print("\nModel:")
print("Random Forest Classifier")

print("\nNumber of trees:")
print("200")

print("\nTraining samples:")
print(len(X_train))

print("\nTesting samples:")
print(len(X_test))

print("\nNumber of features:")
print(len(features))

print("\nNumber of crop classes:")
print(len(label_encoder.classes_))

print("\nCrop classes:")

for crop in label_encoder.classes_:
    print("-", crop)

print("\nSaved files:")

print(
    "✓",
    MODEL_PATH
)

print(
    "✓",
    ENCODER_PATH
)

print("\nYou can now run the Streamlit application.")